# Approximate QFT on 6 Qubits

In this Jupyter notebook, we implement AQFT Circuit 1 from Park-Ahn, with $n = 6$ qubits, error $\varepsilon = 0.2$, and cutoff $b = 5$. We take as a given quantum resource that the ancilla state $\lvert \psi_6 \rvert$ has been prepared correctly.

In [ ]:
from qrisp import QuantumSession, QuantumVariable, QuantumCircuit, prepare, h, z, s, cx, gidney_adder, QFT, barrier

from qrisp import gate_wrap

import matplotlib.pyplot as plt

import numpy as np

In [ ]:
@gate_wrap
def adder(a, b): 
    gidney_adder(a, b)

## Fan Out

The following method applied a fan-out gate as defined in Park-Ahn, which is a sequence of CX gates controlled by a single qubit.

In [ ]:
def fan_out(circuit, control, targets):
    """Fan-out operation using CNOT gates.
    
        Input: circuit -> QuantumCircuit object
               qubit_index -> index of the qubit to fan out from
               ancilla_indices -> list of indices of the ancilla qubits to fan out to

        Output: None
    
    """
    for target in targets:
        circuit.cx(control, target)

## AQFT Circuit

In [ ]:
n = 6                                            # n = number of qubits we are applying the AQFT to
epsilon = 0.2                                      # epsilon = spectral norm error tolerance for the approximation
b = int(np.ceil(np.log2(n/epsilon)))             # b + 1 = number of ancilla qubits needed
print(f"Number of ancilla qubits needed: {b+1}")

We initialize the circuit with one dummy register for the purposes of implementing inverse Phase Gradient Transform using a Gidney adder; with one quantum register on $n$ qubits; and one ancilla register on $b + 1$ qubits.

In [ ]:
qs = QuantumSession()

quantumRegister = QuantumVariable(n, name='q', qs = qs)          # quantum register of n qubits
ancillaRegister = QuantumVariable(int(b+1), name='a', qs = qs)   # ancilla register of b+1 qubits

We assume that we have access to the correct state for ancilla register.

In [ ]:
# This is done by preparing the ancilla register in the state |1>^(b+1) using the prepare function from qrisp.
allOnesAmplitudes = np.array([0]*(2**(b+1) - 1) + [1], dtype=float)
allOnes = prepare(ancillaRegister, allOnesAmplitudes)  # prepare the ancilla register in the state |1>^(b+1)

# We apply inverse Quantum Fourier Transform to the all ones state to prepare the ancilla correctly for use with Gidney adder
QFT(ancillaRegister, inv=True, qiskit_endian=False)  # apply the inverse QFT to the ancilla register

We implement AQFT Circuit 1 from Park-Ahn, Figure 3, for $n = 6$ qubits and $b = 5$.

In [ ]:
h(quantumRegister[0])  # apply Hadamard gate to the first qubit
z(quantumRegister[0])  # apply Z gate to the first qubit

for i in range(1, n):
    s(quantumRegister[i])  # apply S gate to the i-th qubit

for i in range(1,3):
    adder(quantumRegister[0:b], ancillaRegister)    # apply the Gidney adder
    fan_out(qs, quantumRegister[0], quantumRegister[1:b])  # apply the fan-out operation to the first qubit and the ancilla register

barrier(quantumRegister)

for i in range(1, n-1):
    h(quantumRegister[i])  # apply Hadamard gate to the i-th qubit
    s(quantumRegister[i])

    fan_out(qs, quantumRegister[i], quantumRegister[i+1:np.min([i+b,n])])  # apply the fan-out operation to the i-th qubit and the ancilla register
    adder(quantumRegister[i:np.min([i+b,n])], ancillaRegister)  # apply the Gidney adder
    fan_out(qs, quantumRegister[i], quantumRegister[i+1:np.min([i+b,n])])  # apply the fan-out operation to the i-th qubit and the ancilla register again
    barrier(quantumRegister)

h(quantumRegister[n-1])  # apply Hadamard gate to the last qubit

for i in range(n):
    s(quantumRegister[i])  # apply S gate to the i-th qubit

adder(quantumRegister[-1:-b-1:-1], ancillaRegister)  # apply the Gidney adder again

print(qs)

aqft_circuit = qs.to_qiskit()
aqft_circuit.draw(output='mpl', style={'backgroundcolor': '#EEEEEE'}, fold=100, idle_wires=False)

# Future Work

- Testing module implementing Draper's adder and quantum phase estimation using approximate QFT.
- Error quantification in terms of spectral norm